# 04 — Tracking de Objetos

## ¿Qué vamos a construir hoy?

Procesarás un video real y seguirás cada vehículo con un número de identidad
que se mantiene estable aunque el objeto se mueva.

**Aprenderás a:**
- Entender la diferencia entre detectar objetos y seguirlos en el tiempo
- Usar `sv.ByteTrack` para asignar IDs persistentes
- Procesar video con `sv.process_video`

**Tiempo estimado:** 30 minutos

## ⏱️ Estructura de la Clase (Duración estimada: 1 hora)
- **Introducción y Conceptos Base**: 15 min
- **Desarrollo y Demostración Práctica**: 25 min
- **Análisis y Casos Extremos (Pausa y Observa)**: 20 min

## Detección vs. Tracking

Detectar objetos en video es como revisar cada foto de una cámara de seguridad:
sabes cuántos objetos hay en ese instante, pero no sabes si el objeto
del frame 1 es el mismo que el del frame 2.

**El tracker resuelve eso:**
actúa como un guardia con lista de asistencia — compara los objetos nuevos
con los del frame anterior y asigna el mismo número de credencial si los reconoce.

```
Frame N:   detecciones sin ID  ──► tracker ──► detecciones con tracker_id
Frame N+1: detecciones sin ID  ──► tracker ──► mismos tracker_id (si mismo objeto)
```

In [ ]:
!pip install supervision ultralytics
import supervision as sv
from ultralytics import YOLO
import cv2
import numpy as np
import urllib.request
from pathlib import Path

Path("assets").mkdir(exist_ok=True)

print("Descargando video de muestra (puede tardar un momento)...")
urllib.request.urlretrieve(
    "https://media.roboflow.com/supervision/video-examples/vehicles.mp4",
    "assets/vehicles.mp4"
)
print("Video listo.")

video_info = sv.VideoInfo.from_video_path("assets/vehicles.mp4")
print(f"\nResolución: {video_info.width} × {video_info.height} px")
print(f"FPS: {video_info.fps}")
print(f"Duración: {video_info.total_frames / video_info.fps:.1f} segundos")

## Pausa y observa: ¿Qué hace tracker_id ANTES del tracking?

## ¿Por qué usamos `sv.ByteTrack` en este notebook?

`sv.ByteTrack` es el tracker integrado en Supervision. Su ventaja en este notebook
es la API directa: `update_with_detections(det)` acepta y devuelve `sv.Detections`
sin pasos intermedios — ideal para aprender el concepto de tracking.

A partir de Supervision v0.28.0, `sv.ByteTrack` está marcado como **deprecated**
y será removido en v0.30.0, en favor del paquete externo
[`trackers`](https://pypi.org/project/trackers/). La razón del cambio es separar
la lógica de tracking de la librería principal y estandarizar la API entre trackers.

**¿Por qué no migramos aquí?** Porque el objetivo de este notebook es entender
los *conceptos*: qué es un `tracker_id`, cómo persiste entre frames y cómo se
combina con los annotators. `sv.ByteTrack` es suficiente para aprender eso con
menos fricción — la API es simple y el comportamiento es idéntico.

En **NB05** (Zonas) y **NB09** (Video + SAM) usamos el paquete `trackers`
actualizado, ya que en esos notebooks es importante trabajar con la API vigente.

In [ ]:
model = YOLO("yolov8n.pt")

# sv.ByteTrack es la API clásica de Supervision — funciona pero está siendo reemplazada.
# En versiones futuras usarás ByteTrackTracker del paquete 'trackers'.
# Para este tutorial, sv.ByteTrack es suficiente y más directo.
tracker = sv.ByteTrack()

# Inspeccionamos un solo frame para ver el estado ANTES del tracker
cap = cv2.VideoCapture("assets/vehicles.mp4")
ret, primer_frame = cap.read()
cap.release()

results_test = model(primer_frame, verbose=False)[0]
det_sin_tracker = sv.Detections.from_ultralytics(results_test)

print("ANTES de pasar por el tracker:")
print(f"  tracker_id: {det_sin_tracker.tracker_id}")
# tracker_id es None porque nadie ha asignado IDs todavía

det_con_tracker = tracker.update_with_detections(det_sin_tracker)
print("\nDESPUÉS de pasar por el tracker:")
print(f"  tracker_id: {det_con_tracker.tracker_id}")
# Ahora cada detección tiene un número de identificación único

## El pipeline completo con tracking

In [ ]:
# Reiniciamos el tracker para que los IDs empiecen desde 1
# Sin reset(), los IDs continuarían desde donde quedaron en la celda anterior
tracker.reset()

box_annotator   = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()
trace_annotator = sv.TraceAnnotator()
# TraceAnnotator dibuja la trayectoria de cada objeto — necesita tracker_id para funcionar

def procesar_frame(frame: np.ndarray, frame_idx: int) -> np.ndarray:
    results = model(frame, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(results)
    
    # El tracker compara estas detecciones con el frame anterior
    # y asigna el mismo tracker_id si reconoce el objeto
    detections = tracker.update_with_detections(detections)
    
    # Usamos tracker_id (quién es) no class_id (qué tipo de objeto es)
    labels = [f"ID:{tid}" for tid in detections.tracker_id]
    
    annotated = box_annotator.annotate(scene=frame.copy(), detections=detections)
    annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=labels)
    annotated = trace_annotator.annotate(scene=annotated, detections=detections)
    return annotated

sv.process_video(
    source_path="assets/vehicles.mp4",
    target_path="assets/vehicles_tracked.mp4",
    callback=procesar_frame,
    show_progress = True
)
print("Video guardado: assets/vehicles_tracked.mp4")

## 🔧 Exploración interactiva

### Experimento 1: Inspeccionar tracker_id frame a frame

In [ ]:
# Procesamos los primeros 3 frames manualmente para ver cómo evolucionan los IDs
tracker.reset()

cap = cv2.VideoCapture("assets/vehicles.mp4")
for frame_num in range(3):
    ret, frame = cap.read()
    if not ret:
        break
    results = model(frame, verbose=False)[0]
    det = sv.Detections.from_ultralytics(results)
    det = tracker.update_with_detections(det)
    print(f"Frame {frame_num}: {len(det)} objetos | IDs: {det.tracker_id}")
cap.release()
# 💭 Reflexión: ¿Los IDs del frame 0 aparecen también en el frame 1 y 2?
# Si un objeto desaparece y vuelve a aparecer después de varios frames,
# el tracker puede asignarle un ID diferente.

### Experimento 2: Mostrar clase e ID al mismo tiempo

In [ ]:
tracker.reset()

def callback_clase_id(frame: np.ndarray, _: int) -> np.ndarray:
    results = model(frame, verbose=False)[0]
    det = sv.Detections.from_ultralytics(results)
    det = tracker.update_with_detections(det)
    # Combinar el nombre de la clase con el ID del tracker
    # — más informativo que mostrar solo el ID
    labels = [
        f"{results.names[c]} #{tid}"
        for c, tid in zip(det.class_id, det.tracker_id)
    ]
    scene = box_annotator.annotate(scene=frame.copy(), detections=det)
    return label_annotator.annotate(scene=scene, detections=det, labels=labels)

sv.process_video(
    source_path="assets/vehicles.mp4",
    target_path="assets/vehicles_clase_id.mp4",
    callback=callback_clase_id,
    show_progress=True
)
print("Guardado: assets/vehicles_clase_id.mp4")
# 💭 Reflexión: ¿Qué label resulta más útil para tu aplicación:
# solo el ID, solo la clase, o ambos?

### Experimento 3: Filtrar antes de trackear

In [ ]:
# Combinar filtrado (NB03) con tracking — el orden importa:
# filtramos ANTES de pasar al tracker para que no gaste IDs en objetos que no nos interesan
CLASE_OBJETIVO = 2  # 'car' en COCO

tracker.reset()

def callback_solo_autos(frame: np.ndarray, _: int) -> np.ndarray:
    results = model(frame, verbose=False)[0]
    det = sv.Detections.from_ultralytics(results)
    # Filtrar ANTES del tracker — así el tracker solo gestiona la clase de interés
    det = det[det.class_id == CLASE_OBJETIVO]
    det = tracker.update_with_detections(det)
    labels = [f"auto #{tid}" for tid in det.tracker_id]
    scene = box_annotator.annotate(scene=frame.copy(), detections=det)
    return label_annotator.annotate(scene=scene, detections=det, labels=labels)

sv.process_video(
    source_path="assets/vehicles.mp4",
    target_path="assets/vehicles_autos.mp4",
    callback=callback_solo_autos,
    show_progress=True
)
print("Guardado: assets/vehicles_autos.mp4")
# 💭 Reflexión: ¿Por qué conviene filtrar ANTES de pasar al tracker
# en lugar de filtrar DESPUÉS?

## 🚀 Reto de extensión

**Tarea:** Muestra en la etiqueta cuántos frames lleva visible cada objeto.

**Pista:** Crea un diccionario fuera del callback para contar frames por ID:
```python
frame_count = {}

def mi_callback(frame, _):
    # ... detección y tracking ...
    for tid in detections.tracker_id:
        frame_count[tid] = frame_count.get(tid, 0) + 1
    labels = [f"#{tid} ({frame_count[tid]}f)" for tid in detections.tracker_id]
    # ... anotar y retornar ...
```

In [ ]:
# Escribe tu solución aquí
frame_count = {}

tracker.reset()

def mi_callback(frame: np.ndarray, _: int) -> np.ndarray:
    results = model(frame, verbose=False)[0]
    det = sv.Detections.from_ultralytics(results)
    det = tracker.update_with_detections(det)
    # Incrementa el contador y construye las etiquetas aquí
    # labels = ...
    scene = box_annotator.annotate(scene=frame.copy(), detections=det)
    return scene

---

## De `sv.ByteTrack` a `ByteTrackTracker` — lo que cambia en NB05

En NB05 migraremos al paquete `trackers` que es la forma actualizada de usar ByteTrack con Supervision. El cambio es mínimo:

| | NB04 (sv.ByteTrack) | NB05 (trackers) |
|--|---------------------|-----------------|
| **Import** | `tracker = sv.ByteTrack()` | `from trackers import ByteTrackTracker`<br>`tracker = ByteTrackTracker()` |
| **Actualizar** | `tracker.update_with_detections(det)` | `tracker.update(det)` |
| **Reiniciar** | `tracker.reset()` | `tracker = ByteTrackTracker()` |

El pipeline de Supervision no cambia: `sv.Detections`, `BoxAnnotator`, `LabelAnnotator` y `sv.process_video` funcionan exactamente igual.